# Classic DP Paradigms in Machine Learning: 0/1 Knapsack and Edit Distance

**Session:** 2026-11-08 · Dynamic Programming and Divide-and-Conquer  
**Module:** 01 · Python Fundamentals and Algorithmic Thinking  

---

## 1. Overview: Classic DP in Machine Learning Systems

Dynamic Programming solves critical structural and combinatorial problems throughout machine learning:
1. **0/1 Knapsack for Feature Selection / Resource Budgeting:** Selecting the highest-utility feature subset or neural network layer configuration under strict inference memory/latency constraints.
2. **Levenshtein Edit Distance for NLP & Speech:** Aligning ground-truth text with ASR acoustic transcripts to compute Word Error Rate (WER) and Character Error Rate (CER), or correcting OCR errors.


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

DATA_DIR = Path("../data")

# Load Knapsack items
knapsack_items = []
with (DATA_DIR / "knapsack_items.csv").open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        knapsack_items.append(
            {
                "item_id": int(row["item_id"]),
                "feature_name": row["feature_name"],
                "memory_cost_kb": int(row["memory_cost_kb"]),
                "utility_score": int(row["utility_score"]),
            }
        )

# Load Edit Distance pairs
with (DATA_DIR / "edit_distance_pairs.json").open(encoding="utf-8") as f:
    edit_pairs = json.load(f)

print(
    f"Loaded {len(knapsack_items)} candidate features and {len(edit_pairs)} edit distance test pairs."
)

## 2. The 0/1 Knapsack Problem

### Formulation:
Given $n$ features with memory costs $w_1, \dots, w_n$ and utility scores $v_1, \dots, v_n$, choose binary inclusion variables $x_i \in \{0, 1\}$ to maximize total utility subject to capacity $W$:
$$\max \sum_{i=1}^n v_i x_i \quad \text{subject to} \quad \sum_{i=1}^n w_i x_i \le W$$

### Recurrence:
Let $dp[i][w]$ denote the maximum utility attainable using a subset of the first $i$ items with total memory capacity $w$:
$$dp[i][w] = \begin{cases} dp[i-1][w] & \text{if } w_i > w \\ \max(dp[i-1][w], \, dp[i-1][w - w_i] + v_i) & \text{if } w_i \le w \end{cases}$$


In [ ]:
def knapsack_01(weights: list[int], values: list[int], capacity: int) -> tuple[int, list[int]]:
    """2D DP table implementation of 0/1 Knapsack with item recovery."""
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        w = weights[i - 1]
        v = values[i - 1]
        for c in range(capacity + 1):
            if w > c:
                dp[i][c] = dp[i - 1][c]
            else:
                dp[i][c] = max(dp[i - 1][c], dp[i - 1][c - w] + v)

    # Backtrack to reconstruct selected items
    selected = []
    c = capacity
    for i in range(n, 0, -1):
        if dp[i][c] != dp[i - 1][c]:
            selected.append(i - 1)
            c -= weights[i - 1]

    selected.reverse()
    return dp[n][capacity], selected


# Test on ML feature selection with a 75 KB memory budget
weights = [item["memory_cost_kb"] for item in knapsack_items]
values = [item["utility_score"] for item in knapsack_items]
budget = 75

max_util, chosen_indices = knapsack_01(weights, values, budget)
print(f"Optimal Feature Utility under {budget} KB budget: {max_util}")
print("Selected Features:")
for idx in chosen_indices:
    item = knapsack_items[idx]
    print(
        f" - {item['feature_name']} (Memory: {item['memory_cost_kb']} KB, Utility: {item['utility_score']})"
    )

## 3. Levenshtein Edit Distance

### Formulation:
The Levenshtein distance between two strings $s_1$ and $s_2$ is the minimum number of single-character edits (insertions, deletions, or substitutions) required to transform $s_1$ into $s_2$.

### Recurrence:
Let $D(i, j)$ be the edit distance between prefixes $s_1[0:i]$ and $s_2[0:j]$:
$$D(i, 0) = i, \quad D(0, j) = j$$
$$D(i, j) = \begin{cases} D(i-1, j-1) & \text{if } s_1[i-1] == s_2[j-1] \\ 1 + \min \begin{cases} D(i-1, j) & \text{(Delete)} \\ D(i, j-1) & \text{(Insert)} \\ D(i-1, j-1) & \text{(Substitute)} \end{cases} & \text{if } s_1[i-1] \ne s_2[j-1] \end{cases}$$


In [ ]:
def levenshtein_distance(s1: str, s2: str) -> tuple[int, list[list[int]]]:
    """Compute Levenshtein edit distance matrix between two strings."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],  # Deletion
                    dp[i][j - 1],  # Insertion
                    dp[i - 1][j - 1],  # Substitution
                )

    return dp[m][n], dp


def format_alignment(s1: str, s2: str, dp: list[list[int]]) -> list[str]:
    """Reconstruct the sequence of edit operations from the DP matrix."""
    i, j = len(s1), len(s2)
    operations = []

    while i > 0 or j > 0:
        if i > 0 and j > 0 and s1[i - 1] == s2[j - 1]:
            operations.append(f"Match '{s1[i - 1]}'")
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            operations.append(f"Substitute '{s1[i - 1]}' -> '{s2[j - 1]}'")
            i -= 1
            j -= 1
        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
            operations.append(f"Insert '{s2[j - 1]}'")
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            operations.append(f"Delete '{s1[i - 1]}'")
            i -= 1

    operations.reverse()
    return operations


dist, matrix = levenshtein_distance("kitten", "sitting")
ops = format_alignment("kitten", "sitting", matrix)
print(f"Edit distance between 'kitten' and 'sitting': {dist}")
print("Operations:")
for op in ops:
    print(" -", op)

## 4. Automated Testing on NLP Test Pairs


In [ ]:
for case in edit_pairs:
    s1 = case["str1"]
    s2 = case["str2"]
    expected = case["expected_distance"]
    computed, _ = levenshtein_distance(s1, s2)
    assert computed == expected, f"Mismatch on {case['id']}: got {computed}, expected {expected}"
    print(f"PASS: {case['id']:<10} | '{s1}' vs '{s2}' -> Distance: {computed}")